In [ ]:
!sudo apt install tesseract-ocr
!pip install pytesseract
!pip install pdf2image
!apt-get install poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  tesseract-ocr-eng tesseract-ocr-osd
The following NEW packages will be installed:
  tesseract-ocr tesseract-ocr-eng tesseract-ocr-osd
0 upgraded, 3 newly installed, 0 to remove and 49 not upgraded.
Need to get 4,816 kB of archives.
After this operation, 15.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-eng all 1:4.00~git30-7274cfa-1.1 [1,591 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-osd all 1:4.00~git30-7274cfa-1.1 [2,990 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr amd64 4.1.1-2.1build1 [236 kB]
Fetched 4,816 kB in 2s (2,846 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debc

In [ ]:
from pdf2image import convert_from_path
from google.colab import files
import cv2
import numpy as np
import pandas as pd
import pytesseract
import re

# Upload the PDF file
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
with open(pdf_filename, 'wb') as f:
    f.write(uploaded[pdf_filename])

# Convert PDF to images
pages = convert_from_path(pdf_filename)

# Deskew function
def deskew(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.bitwise_not(gray)
    coords = np.column_stack(np.where(gray > 0))
    angle = cv2.minAreaRect(coords)[-1]

    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle

    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    return rotated

# Extract text using Tesseract OCR
def extract_text_from_image(image):
    return pytesseract.image_to_string(image)

# Preprocess and extract text from all pages
extracted_text = []
for page in pages:
    preprocessed_image = deskew(np.array(page))
    text = extract_text_from_image(preprocessed_image)
    extracted_text.append(text)

# Combine text from all pages
resume_text = "".join(extracted_text)

# Load skills dataset and normalize to lowercase for uniformity
skill_data = pd.read_excel('/content/drive/MyDrive/major_project/skills_dataset.xlsx')  # Replace with actual path
skills_list = skill_data['Skills'].str.lower().unique().tolist()

# Function to capture multi-word skills
def extract_skills(text, skills_list):
    skill_headers = ['SKILLS', 'TECHNICAL SKILLS', 'SKILL SET', 'EXPERTISE']
    skills_section = None

    # Locate the skills section in the resume text
    for header in skill_headers:
        if header in text:
            skills_section = text.split(header, 1)[1]
            break

    # Initialize the extracted skills list
    extracted_skills = []
    if skills_section:
        lines = skills_section.splitlines()
        for line in lines:
            if re.search(r'^[A-Z][A-Z\s]*$', line.strip()):
                break
            # Clean the line to remove special characters and unnecessary spaces
            line = re.sub(r'[^\w\s]', '', line).strip().lower()

            # Check for skills in each line, allowing multi-word skill matching
            for skill in skills_list:
                skill_pattern = re.escape(skill)
                if re.search(r'\b' + skill_pattern + r'\b', line, re.IGNORECASE):
                    extracted_skills.append(skill)

    # Deduplicate the extracted skills
    return list(set(extracted_skills))

# Extracted skills
skills_found = extract_skills(resume_text, skills_list)
print("Extracted Skills:")
for skill in skills_found:
    print(skill)


Saving SanikaPatankar_Resume.pdf to SanikaPatankar_Resume.pdf
Extracted Skills:
python
html
javascript
tensorflow
c
java
django
machine learning
css
sql


# **S3M**

In [ ]:
from sentence_transformers import SentenceTransformer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Dot, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Initialize the pre-trained Sentence-BERT model
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Paths to datasets
datasets = {
    "Dataset 1": "/content/drive/MyDrive/major_project/dataset_1.csv",
    "Dataset 2": "/content/drive/MyDrive/major_project/dataset_2.csv",
    "Dataset 3": "/content/drive/MyDrive/major_project/dataset_3.csv",
    "Dataset 4" : "/content/drive/MyDrive/major_project/dataset_4.csv",
    "Dataset 5" : "/content/drive/MyDrive/major_project/dataset_5.csv",
    "Dataset 6" : "/content/drive/MyDrive/major_project/dataset_6.csv"
}

# Load dataset function
def load_dataset(file_path):
    df = pd.read_csv(file_path)
    job_titles = df['Job Title'].values
    job_descriptions = df['Job Description'].values
    return job_titles, job_descriptions

# Load data and prepare Sentence-BERT embeddings
all_job_titles = []
all_job_descriptions = []

for dataset_path in datasets.values():
    job_titles, job_descriptions = load_dataset(dataset_path)
    all_job_titles.extend(job_titles)
    all_job_descriptions.extend(job_descriptions)

# Generate Sentence-BERT embeddings
job_titles_embeddings = sbert_model.encode(all_job_titles, convert_to_tensor=True)
job_descriptions_embeddings = sbert_model.encode(all_job_descriptions, convert_to_tensor=True)

# Create Positive and Negative Pairs for Self-Supervised Learning
positive_pairs = list(zip(job_titles_embeddings, job_descriptions_embeddings))
negative_pairs = []

# Generate random negative pairs (randomly mismatched job titles and descriptions)
for _ in range(len(positive_pairs)):
    i, j = np.random.choice(len(all_job_titles), 2, replace=False)
    negative_pairs.append((job_titles_embeddings[i], job_descriptions_embeddings[j]))

# Prepare labels: 1 for positive pairs, 0 for negative pairs
labels = np.array([1] * len(positive_pairs) + [0] * len(negative_pairs))

# Concatenate positive and negative pairs
title_embeds = np.array([pair[0] for pair in positive_pairs + negative_pairs])
description_embeds = np.array([pair[1] for pair in positive_pairs + negative_pairs])

# Split data into training and testing sets
titles_train, titles_test, descriptions_train, descriptions_test, labels_train, labels_test = train_test_split(
    title_embeds, description_embeds, labels, test_size=0.2, random_state=42
)

# Define cosine similarity model in Keras
input_title = Input(shape=(job_titles_embeddings.shape[1],))
input_description = Input(shape=(job_descriptions_embeddings.shape[1],))

# Dense layers for feature extraction
dense_title = Dense(256, activation='relu')(input_title)
dense_title = Dropout(0.4)(dense_title)
dense_title = Dense(128, activation='relu')(dense_title)

dense_description = Dense(256, activation='relu')(input_description)
dense_description = Dropout(0.4)(dense_description)
dense_description = Dense(128, activation='relu')(dense_description)

# Cosine similarity layer
cosine_sim = Dot(axes=1, normalize=True)([dense_title, dense_description])

# Compile the model
model = Model(inputs=[input_title, input_description], outputs=cosine_sim)
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# Train the model with EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(
    [titles_train, descriptions_train], labels_train,
    epochs=15, batch_size=32, validation_split=0.2, callbacks=[early_stopping], verbose=1
)

# Evaluate the model
val_loss, val_accuracy = model.evaluate([titles_test, descriptions_test], labels_test)
print(f"Validation Accuracy: {val_accuracy:.4f}")

# Function to predict top 3 job titles for a given skillset
def predict_top_3_jobs(skillset_text):
    # Get embedding for the skillset
    skillset_embedding = sbert_model.encode([skillset_text])

    # Calculate similarity scores between skillset and each job title embedding
    similarity_scores = np.dot(job_titles_embeddings, skillset_embedding.T).flatten()

    # Get top 3 job titles based on similarity scores
    top_3_indices = np.argsort(similarity_scores)[-3:][::-1]
    top_3_job_titles = [all_job_titles[i] for i in top_3_indices]
    top_3_scores = similarity_scores[top_3_indices]

    # Print the top 3 job titles and their similarity scores
    print(f"Top 3 Jobs for skillset '{skillset_text}':")
    for i, (job, score) in enumerate(zip(top_3_job_titles, top_3_scores)):
        print(f"{i+1}. {job} - Similarity Score: {score:.2f}")

# Example Usage
sample_skillset = str(skills_found)
predict_top_3_jobs(sample_skillset)


Epoch 1/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 23s 9ms/step - accuracy: 0.6048 - loss: 0.6532 - val_accuracy: 0.7233 - val_loss: 0.5339
Epoch 2/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 39s 8ms/step - accuracy: 0.7330 - loss: 0.5321 - val_accuracy: 0.7655 - val_loss: 0.4807
Epoch 3/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.7656 - loss: 0.4900 - val_accuracy: 0.7841 - val_loss: 0.4576
Epoch 4/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - accuracy: 0.7821 - loss: 0.4660 - val_accuracy: 0.7929 - val_loss: 0.4436
Epoch 5/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - accuracy: 0.7929 - loss: 0.4511 - val_accuracy: 0.7983 - val_loss: 0.4359
Epoch 6/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.8038 - loss: 0.4318 - val_accuracy: 0.8020 - val_loss: 0.4289
Epoch 7/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 42s 9ms/step - accuracy: 0.8091 - loss: 0.4218 - val_accuracy: 0.8056 - val_loss: 0.4269
Epoch 8/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 38s 8ms/step - accuracy: 0.8160 - loss: 0

## DSSM

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Concatenate, Input, Dot, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
from sklearn.model_selection import train_test_split

# Paths to datasets
datasets = {
    "Dataset 1": "/content/drive/MyDrive/major_project/dataset_1.csv",
    "Dataset 2": "/content/drive/MyDrive/major_project/dataset_2.csv",
    "Dataset 3": "/content/drive/MyDrive/major_project/dataset_3.csv",
    "Dataset 4" : "/content/drive/MyDrive/major_project/dataset_4.csv",
    "Dataset 5" : "/content/drive/MyDrive/major_project/dataset_5.csv",
    "Dataset 6" : "/content/drive/MyDrive/major_project/dataset_6.csv"
}

# Load dataset function
def load_dataset(file_path):
    df = pd.read_csv(file_path)
    job_titles = df['Job Title'].values
    job_descriptions = df['Job Description'].values
    return job_titles, job_descriptions

# Step 1: Load data from all datasets and combine for TF-IDF fitting
all_job_titles = []
all_job_descriptions = []

for dataset_path in datasets.values():
    job_titles, job_descriptions = load_dataset(dataset_path)
    all_job_titles.extend(job_titles)
    all_job_descriptions.extend(job_descriptions)

# Step 2: Fit TF-IDF Vectorizer on all job titles and descriptions combined
max_features = 5000
vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
vectorizer.fit(np.concatenate((all_job_titles, all_job_descriptions), axis=0))

# Step 3: Transform Job Titles and Descriptions for each dataset
job_titles_tfidf = vectorizer.transform(all_job_titles).toarray()
job_descriptions_tfidf = vectorizer.transform(all_job_descriptions).toarray()

# Step 4: Improved DSSM Model with Cosine Similarity and Dense Layers
input_job_title = Input(shape=(max_features,))
input_job_description = Input(shape=(max_features,))

# Dense layers for job title and description
title_dense = Dense(256, activation='relu')(input_job_title)
title_dense = Dropout(0.5)(title_dense)
title_dense = Dense(128, activation='relu')(title_dense)

description_dense = Dense(256, activation='relu')(input_job_description)
description_dense = Dropout(0.5)(description_dense)
description_dense = Dense(128, activation='relu')(description_dense)

# Cosine similarity between job title and description embeddings
cosine_sim = Dot(axes=1, normalize=True)([title_dense, description_dense])

# Final DSSM model with cosine similarity as output
model = Model(inputs=[input_job_title, input_job_description], outputs=cosine_sim)
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Step 5: Prepare Data for Training (Using dummy binary labels for matching example)
labels = np.random.randint(0, 2, len(all_job_titles))  # Replace with actual labels if available

# Split into training and test sets
job_titles_train, job_titles_test, job_descriptions_train, job_descriptions_test, labels_train, labels_test = train_test_split(
    job_titles_tfidf, job_descriptions_tfidf, labels, test_size=0.2, random_state=42
)

# Train the model
history = model.fit(
    [job_titles_train, job_descriptions_train], labels_train,
    epochs=10, batch_size=32, validation_split=0.2, verbose=1
)

# Evaluate the model
val_loss, val_accuracy = model.evaluate([job_titles_test, job_descriptions_test], labels_test)
print(f"Validation Accuracy: {val_accuracy:.4f}")

# Function to predict top 3 job titles for a given skillset using cosine similarity
def predict_top_3_jobs(skillset_text):
    # Transform the skillset text using the vectorizer
    skillset_tfidf = vectorizer.transform([skillset_text]).toarray()

    # Transform all job titles using the vectorizer
    job_titles_tfidf = vectorizer.transform(all_job_titles).toarray()

    # Predict cosine similarity scores for each job title
    similarity_scores = model.predict([job_titles_tfidf, np.repeat(skillset_tfidf, len(all_job_titles), axis=0)])

    # Get the top 3 job titles based on similarity scores
    top_3_indices = similarity_scores.flatten().argsort()[-3:][::-1]
    top_3_job_titles = np.array(all_job_titles)[top_3_indices]
    top_3_scores = similarity_scores[top_3_indices]

    print(f"Top 3 Jobs for skillset '{skillset_text}':")
    for i, (job, score) in enumerate(zip(top_3_job_titles, top_3_scores)):
        print(f"{i+1}. {job} - Similarity Score: {score[0]:.2f}")

# Example Usage
sample_skillset = str(skills_found)
predict_top_3_jobs(sample_skillset)


Epoch 1/10
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 45s 37ms/step - accuracy: 0.4946 - loss: 0.7078 - val_accuracy: 0.5092 - val_loss: 0.7022
Epoch 2/10
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 78s 34ms/step - accuracy: 0.5420 - loss: 0.6891 - val_accuracy: 0.5052 - val_loss: 0.7000
Epoch 3/10
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 43s 35ms/step - accuracy: 0.5837 - loss: 0.6671 - val_accuracy: 0.5028 - val_loss: 0.7203
Epoch 4/10
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 42s 36ms/step - accuracy: 0.6214 - loss: 0.6385 - val_accuracy: 0.5057 - val_loss: 0.7453
Epoch 5/10
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 83s 36ms/step - accuracy: 0.6565 - loss: 0.6010 - val_accuracy: 0.5038 - val_loss: 0.7917
Epoch 6/10
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 84s 38ms/step - accuracy: 0.6891 - loss: 0.5661 - val_accuracy: 0.5081 - val_loss: 0.8090
Epoch 7/10
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 79s 35ms/step - accuracy: 0.7092 - loss: 0.5359 - val_accuracy: 0.5037 - val_loss: 0.8586
Epoch 8/10
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 43s 37ms/step - accuracy: 0.7249 -

# **DNN**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, Embedding, Flatten, Concatenate
from tensorflow.keras.optimizers import RMSprop
from sklearn.model_selection import train_test_split

# Paths to datasets
datasets = {
    "Dataset 1": "/content/drive/MyDrive/major_project/dataset_1.csv",
    "Dataset 2": "/content/drive/MyDrive/major_project/dataset_2.csv",
    "Dataset 3": "/content/drive/MyDrive/major_project/dataset_3.csv",
    "Dataset 4": "/content/drive/MyDrive/major_project/dataset_4.csv",
    "Dataset 5": "/content/drive/MyDrive/major_project/dataset_5.csv",
    "Dataset 6": "/content/drive/MyDrive/major_project/dataset_6.csv"
}

# Load dataset function
def load_dataset(file_path):
    df = pd.read_csv(file_path)
    job_titles = df['Job Title'].values
    job_descriptions = df['Job Description'].values
    return job_titles, job_descriptions

# Step 1: Load data from all datasets
all_job_titles = []
all_job_descriptions = []

for dataset_path in datasets.values():
    job_titles, job_descriptions = load_dataset(dataset_path)
    all_job_titles.extend(job_titles)
    all_job_descriptions.extend(job_descriptions)

# Step 2: Vectorize job descriptions using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
job_descriptions_tfidf = vectorizer.fit_transform(all_job_descriptions).toarray()
actual_feature_size = job_descriptions_tfidf.shape[1]

# Step 3: Convert job titles to indices
title_to_index = {title: idx for idx, title in enumerate(set(all_job_titles))}
job_titles_indices = [title_to_index[title] for title in all_job_titles]

# Step 4: Define DNN model
input_job_title = Input(shape=(1,))
input_job_description = Input(shape=(actual_feature_size,))

# Embedding layer for job title
title_embedding = Embedding(input_dim=len(title_to_index), output_dim=50)(input_job_title)
title_embedding = Flatten()(title_embedding)

# Dense layers for job description context features
description_dense = Dense(512, activation='relu')(input_job_description)
description_dense = Dropout(0.5)(description_dense)
description_dense = Dense(256, activation='relu')(description_dense)
description_dense = Dropout(0.5)(description_dense)

# Merge categorical and context features
merged = Concatenate()([title_embedding, description_dense])

# Additional dense layers for the merged features
final_dense = Dense(256, activation='relu')(merged)
final_dense = Dropout(0.5)(final_dense)
final_dense = Dense(128, activation='relu')(final_dense)

# Output layer for binary classification
output = Dense(1, activation='sigmoid')(final_dense)

# Compile model
model = Model(inputs=[input_job_title, input_job_description], outputs=output)
model.compile(optimizer=RMSprop(), loss='binary_crossentropy', metrics=['accuracy'])

# Step 5: Convert data for model training
job_titles_np = np.array(job_titles_indices).reshape(-1, 1)
job_descriptions_np = np.array(job_descriptions_tfidf)
labels = np.random.randint(0, 2, len(job_titles_np))  # Replace with actual labels if available

# Split data into training and test sets
job_titles_train, job_titles_test, job_descriptions_train, job_descriptions_test, labels_train, labels_test = train_test_split(
    job_titles_np, job_descriptions_np, labels, test_size=0.2, random_state=42
)

# Step 6: Train the model
history = model.fit(
    [job_titles_train, job_descriptions_train], labels_train,
    epochs=5, batch_size=32, validation_split=0.2, verbose=1
)

# Step 7: Evaluate the model
val_loss, val_accuracy = model.evaluate([job_titles_test, job_descriptions_test], labels_test)
print(f"Validation Accuracy: {val_accuracy:.4f}")

# Step 8: Function to predict top 3 job titles for a given skillset
def predict_top_3_jobs(skillset_text):
    # Transform the skillset text using the vectorizer
    skillset_tfidf = vectorizer.transform([skillset_text]).toarray()

    # Use valid indices from title_to_index
    valid_indices = np.array(list(title_to_index.values())).reshape(-1, 1)

    # Predict similarity scores for all valid job titles
    similarity_scores = model.predict([valid_indices, np.repeat(skillset_tfidf, len(valid_indices), axis=0)])

    # Get the top 3 job titles based on similarity scores
    top_3_indices = np.argsort(similarity_scores.flatten())[-3:][::-1]
    top_3_job_titles = np.array(list(title_to_index.keys()))[top_3_indices]
    top_3_scores = similarity_scores[top_3_indices]

    print(f"Top 3 Jobs for skillset '{skillset_text}':")
    for i, (job, score) in enumerate(zip(top_3_job_titles, top_3_scores)):
        print(f"{i + 1}. {job} - Similarity Score: {score[0]:.2f}")

# Example Usage
sample_skillset = "machine learning, deep learning, Python"
predict_top_3_jobs(sample_skillset)



Epoch 1/5
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 36s 28ms/step - accuracy: 0.5059 - loss: 0.6941 - val_accuracy: 0.5007 - val_loss: 0.6931
Epoch 2/5
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 31s 27ms/step - accuracy: 0.5141 - loss: 0.6928 - val_accuracy: 0.4986 - val_loss: 0.6945
Epoch 3/5
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 43s 28ms/step - accuracy: 0.6102 - loss: 0.6710 - val_accuracy: 0.5006 - val_loss: 0.7413
Epoch 4/5
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 41s 28ms/step - accuracy: 0.7363 - loss: 0.5635 - val_accuracy: 0.4957 - val_loss: 0.8347
Epoch 5/5
1171/1171 ━━━━━━━━━━━━━━━━━━━━ 45s 31ms/step - accuracy: 0.7724 - loss: 0.4805 - val_accuracy: 0.4923 - val_loss: 0.8905
366/366 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4993 - loss: 0.8926
Validation Accuracy: 0.4985
955/955 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step
Top 3 Jobs for skillset 'machine learning, deep learning, Python':
1. Python Engineer - Similarity Score: 0.90
2.  SQL Developer - Similarity Score: 0.90
3. Senior Researcher - Similarity Score: 0.90


# **DJM**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Input, Dot, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from scipy.sparse import vstack

# Load datasets in batches
def load_dataset(file_path):
    chunk = pd.read_csv(file_path)
    chunk.drop_duplicates(subset=['Job Title', 'Job Description'], inplace=True)
    chunk['Job Title'] = chunk['Job Title'].fillna('')
    chunk['Job Description'] = chunk['Job Description'].fillna('')
    return chunk['Job Title'].tolist(), chunk['Job Description'].tolist()

# Combine all datasets
datasets = {
    "Dataset 1": "/content/drive/MyDrive/major_project/dataset_1.csv",
    "Dataset 2": "/content/drive/MyDrive/major_project/dataset_2.csv",
    "Dataset 3": "/content/drive/MyDrive/major_project/dataset_3.csv",
    "Dataset 4": "/content/drive/MyDrive/major_project/dataset_4.csv",
    "Dataset 5": "/content/drive/MyDrive/major_project/dataset_5.csv",
    "Dataset 6": "/content/drive/MyDrive/major_project/dataset_6.csv"
}

all_job_titles, all_job_descriptions = [], []
for path in datasets.values():
    titles, descriptions = load_dataset(path)
    all_job_titles.extend(titles)
    all_job_descriptions.extend(descriptions)

# TF-IDF Vectorizer
max_features = 5000
vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
vectorizer.fit(all_job_titles + all_job_descriptions)

job_titles_tfidf = vectorizer.transform(all_job_titles)
job_descriptions_tfidf = vectorizer.transform(all_job_descriptions)

# Define the model
input_title = Input(shape=(max_features,))
input_description = Input(shape=(max_features,))

# Dense layers for job title and description
title_dense = Dense(256, activation='relu')(input_title)
title_dense = BatchNormalization()(title_dense)
title_dense = Dropout(0.3)(title_dense)

desc_dense = Dense(256, activation='relu')(input_description)
desc_dense = BatchNormalization()(desc_dense)
desc_dense = Dropout(0.3)(desc_dense)

# Cosine similarity output
cosine_similarity = Dot(axes=1, normalize=True)([title_dense, desc_dense])

# Compile the model
model = Model(inputs=[input_title, input_description], outputs=cosine_similarity)
model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])

# Generate random labels for demonstration (replace with actual labels)
labels = np.random.randint(0, 2, len(all_job_titles))

# Split data for training and testing
X_train_title, X_test_title, X_train_desc, X_test_desc, y_train, y_test = train_test_split(
    job_titles_tfidf, job_descriptions_tfidf, labels, test_size=0.2, random_state=42
)

# Train the model
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

history = model.fit(
    [X_train_title.toarray(), X_train_desc.toarray()], y_train,
    validation_split=0.2, epochs=5, batch_size=32,
    callbacks=[early_stopping, lr_scheduler]
)

# Evaluate the model
loss, accuracy = model.evaluate([X_test_title.toarray(), X_test_desc.toarray()], y_test)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

# Predict Top 3 Job Titles
def predict_top_3_jobs(skillset_text, batch_size=1000):
    skillset_tfidf = vectorizer.transform([skillset_text])
    num_titles = job_titles_tfidf.shape[0]
    similarity_scores = []

    for start_idx in range(0, num_titles, batch_size):
        end_idx = min(start_idx + batch_size, num_titles)
        batch_titles = job_titles_tfidf[start_idx:end_idx]
        batch_scores = model.predict([batch_titles.toarray(), np.repeat(skillset_tfidf.toarray(), batch_titles.shape[0], axis=0)])
        similarity_scores.extend(batch_scores.flatten())

    similarity_scores = np.array(similarity_scores)
    top_3_indices = similarity_scores.argsort()[-3:][::-1]
    top_3_jobs = np.array(all_job_titles)[top_3_indices]
    top_3_scores = similarity_scores[top_3_indices]

    print("\nTop 3 Job Titles:")
    for i, (job, score) in enumerate(zip(top_3_jobs, top_3_scores), 1):
        print(f"{i}. {job} - Similarity Score: {score:.4f}")

# Example Usage
sample_skillset = "Python, Machine Learning, Data Analysis"
predict_top_3_jobs(sample_skillset)


Epoch 1/5
891/891 ━━━━━━━━━━━━━━━━━━━━ 42s 44ms/step - accuracy: 0.4983 - loss: 3.9467 - val_accuracy: 0.4995 - val_loss: 2.5550 - learning_rate: 5.0000e-04
Epoch 2/5
891/891 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.5043 - loss: 2.0036 - val_accuracy: 0.4989 - val_loss: 2.1319 - learning_rate: 5.0000e-04
Epoch 3/5
891/891 ━━━━━━━━━━━━━━━━━━━━ 44s 44ms/step - accuracy: 0.5004 - loss: 1.5864 - val_accuracy: 0.4984 - val_loss: 1.9075 - learning_rate: 5.0000e-04
Epoch 4/5
891/891 ━━━━━━━━━━━━━━━━━━━━ 39s 41ms/step - accuracy: 0.5007 - loss: 1.3061 - val_accuracy: 0.4974 - val_loss: 1.8293 - learning_rate: 5.0000e-04
Epoch 5/5
891/891 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.4987 - loss: 1.1514 - val_accuracy: 0.4991 - val_loss: 1.6992 - learning_rate: 5.0000e-04
279/279 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5005 - loss: 1.6905
Test Loss: 1.7009, Test Accuracy: 0.4933
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
32/32 ━━━━━━━━━━━━━━

In [ ]:
# import os
# from sentence_transformers import SentenceTransformer
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Dense, Dropout, Dot, Input
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import EarlyStopping
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split

# # Initialize the pre-trained Sentence-BERT model
# sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# # Paths to datasets
# datasets = {
#     "Dataset 1": "/content/drive/MyDrive/major_project/dataset_1.csv",
#     "Dataset 2": "/content/drive/MyDrive/major_project/dataset_2.csv",
#     "Dataset 3": "/content/drive/MyDrive/major_project/dataset_3.csv",
#     "Dataset 4": "/content/drive/MyDrive/major_project/dataset_4.csv",
#     "Dataset 5": "/content/drive/MyDrive/major_project/dataset_5.csv",
#     "Dataset 6": "/content/drive/MyDrive/major_project/dataset_6.csv"
# }

# # Directory to save/load embeddings
# embedding_dir = "/content/embeddings/"
# os.makedirs(embedding_dir, exist_ok=True)

# # Load dataset function
# def load_dataset(file_path):
#     df = pd.read_csv(file_path)
#     job_titles = df['Job Title'].values
#     job_descriptions = df['Job Description'].values
#     return job_titles, job_descriptions

# # Function to save embeddings
# def save_embeddings(embeddings, file_path):
#     np.save(file_path, embeddings)

# # Function to load embeddings
# def load_embeddings(file_path):
#     return np.load(file_path)

# # Generate or load Sentence-BERT embeddings
# all_job_titles = []
# all_job_descriptions = []

# for dataset_name, dataset_path in datasets.items():
#     job_titles, job_descriptions = load_dataset(dataset_path)
#     all_job_titles.extend(job_titles)
#     all_job_descriptions.extend(job_descriptions)

# job_titles_file = os.path.join(embedding_dir, "job_titles_embeddings.npy")
# job_descriptions_file = os.path.join(embedding_dir, "job_descriptions_embeddings.npy")

# if os.path.exists(job_titles_file) and os.path.exists(job_descriptions_file):
#     job_titles_embeddings = load_embeddings(job_titles_file)
#     job_descriptions_embeddings = load_embeddings(job_descriptions_file)
#     print("Loaded embeddings from saved files.")
# else:
#     job_titles_embeddings = sbert_model.encode(all_job_titles, convert_to_tensor=False)
#     job_descriptions_embeddings = sbert_model.encode(all_job_descriptions, convert_to_tensor=False)
#     save_embeddings(job_titles_embeddings, job_titles_file)
#     save_embeddings(job_descriptions_embeddings, job_descriptions_file)
#     print("Generated and saved new embeddings.")

# # Create Positive and Negative Pairs for Self-Supervised Learning
# positive_pairs = list(zip(job_titles_embeddings, job_descriptions_embeddings))
# negative_pairs = []

# # Generate random negative pairs (randomly mismatched job titles and descriptions)
# for _ in range(len(positive_pairs)):
#     i, j = np.random.choice(len(all_job_titles), 2, replace=False)
#     negative_pairs.append((job_titles_embeddings[i], job_descriptions_embeddings[j]))

# # Prepare labels: 1 for positive pairs, 0 for negative pairs
# labels = np.array([1] * len(positive_pairs) + [0] * len(negative_pairs))

# # Concatenate positive and negative pairs
# title_embeds = np.array([pair[0] for pair in positive_pairs + negative_pairs])
# description_embeds = np.array([pair[1] for pair in positive_pairs + negative_pairs])

# # Split data into training and testing sets
# titles_train, titles_test, descriptions_train, descriptions_test, labels_train, labels_test = train_test_split(
#     title_embeds, description_embeds, labels, test_size=0.2, random_state=42
# )

# # Define cosine similarity model in Keras
# input_title = Input(shape=(job_titles_embeddings.shape[1],))
# input_description = Input(shape=(job_descriptions_embeddings.shape[1],))

# # Dense layers for feature extraction
# dense_title = Dense(256, activation='relu')(input_title)
# dense_title = Dropout(0.4)(dense_title)
# dense_title = Dense(128, activation='relu')(dense_title)

# dense_description = Dense(256, activation='relu')(input_description)
# dense_description = Dropout(0.4)(dense_description)
# dense_description = Dense(128, activation='relu')(dense_description)

# # Cosine similarity layer
# cosine_sim = Dot(axes=1, normalize=True)([dense_title, dense_description])

# # Compile the model
# model = Model(inputs=[input_title, input_description], outputs=cosine_sim)
# model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# # Train the model with EarlyStopping
# early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
# history = model.fit(
#     [titles_train, descriptions_train], labels_train,
#     epochs=15, batch_size=32, validation_split=0.2, callbacks=[early_stopping], verbose=1
# )

# # Evaluate the model
# val_loss, val_accuracy = model.evaluate([titles_test, descriptions_test], labels_test)
# print(f"Validation Accuracy: {val_accuracy:.4f}")

# # Function to predict top 3 job titles for a given skillset
# def predict_top_3_jobs(skillset_text):
#     skillset_embedding = sbert_model.encode([skillset_text])
#     similarity_scores = np.dot(job_titles_embeddings, skillset_embedding.T).flatten()

#     top_3_indices = np.argsort(similarity_scores)[-3:][::-1]
#     top_3_job_titles = [all_job_titles[i] for i in top_3_indices]
#     top_3_scores = similarity_scores[top_3_indices]

#     print(f"Top 3 Jobs for skillset '{skillset_text}':")
#     for i, (job, score) in enumerate(zip(top_3_job_titles, top_3_scores)):
#         print(f"{i+1}. {job} - Similarity Score: {score:.2f}")

# # Example Usage
# sample_skillset = "Python, Machine Learning, Data Analysis"
# predict_top_3_jobs(sample_skillset)


/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generated and saved new embeddings.
Epoch 1/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 23s 9ms/step - accuracy: 0.6092 - loss: 0.6507 - val_accuracy: 0.7394 - val_loss: 0.5199
Epoch 2/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 23s 10ms/step - accuracy: 0.7382 - loss: 0.5310 - val_accuracy: 0.7718 - val_loss: 0.4768
Epoch 3/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - accuracy: 0.7680 - loss: 0.4893 - val_accuracy: 0.7841 - val_loss: 0.4582
Epoch 4/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 41s 9ms/step - accuracy: 0.7821 - loss: 0.4654 - val_accuracy: 0.7913 - val_loss: 0.4433
Epoch 5/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - accuracy: 0.7930 - loss: 0.4486 - val_accuracy: 0.7985 - val_loss: 0.4326
Epoch 6/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.8037 - loss: 0.4341 - val_accuracy: 0.8049 - val_loss: 0.4224
Epoch 7/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.8093 - loss: 0.4244 - val_accuracy: 0.8078 - val_loss: 0.4187
Epoch 8/15
2342/2342 ━━━━━━━━━━━━━━━━━━━━ 20s 